# Thực nghiệm PhoBERT: so sánh đầu vào và cắt token

Notebook này lấy dữ liệu từ Kaggle Dataset `phuocthoai/stock-trend-forecasting`, chạy `sentiment-cv` trên Google Colab hoặc Kaggle GPU, và lưu checkpoint/kết quả vào vùng lưu trữ tương ứng. Quy trình dùng 5-fold cross-validation với outer holdout chỉ để đánh giá.

## Trước khi chạy

1. Trên Colab, chọn **Runtime → Change runtime type → T4 GPU**; trên Kaggle, chọn GPU trong Settings.
2. Dataset sử dụng: `phuocthoai/stock-trend-forecasting`. Trên Kaggle, có thể Attach Dataset; trên Colab, cell tải dữ liệu bằng `kagglehub`.
3. Dataset phải có `to_label_r1.csv` gồm `title`, `body_preview`, `published_at`, `url`, `label`; tệp phải có ít nhất 301 nhãn cuối đã được một người rà soát xác nhận.
4. Không đưa nhãn sơ bộ vào kết quả báo cáo.

In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/nphuoctho/stock-trend-forecasting.git"
BRANCH = "develop"
REPO_DIR = Path("/content/stock-trend-forecasting")
if Path("/kaggle").exists():
    REPO_DIR = Path("/kaggle/working/stock-trend-forecasting")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
os.chdir(REPO_DIR)
SRC_DIR = REPO_DIR / "src"
if not SRC_DIR.is_dir():
    raise FileNotFoundError(f"Không tìm thấy source package: {SRC_DIR}")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
SUBPROCESS_ENV = {
    **os.environ,
    "PYTHONPATH": os.pathsep.join(
        part for part in (str(SRC_DIR), os.environ.get("PYTHONPATH", "")) if part
    ),
}
HF_PKGS = [
    "transformers==5.15.1",
    "tokenizers>=0.22,<=0.23.0",
    "huggingface-hub>=1.5,<2.0",
    "accelerate>=1.1,<2",
    "sentencepiece>=0.2,<1",
]
if importlib.util.find_spec("torchvision") is None:
    print("torchvision is not installed; text-only PhoBERT training continues.")
else:
    torchvision_check = subprocess.run(
        [sys.executable, "-c", "import torchvision"],
        capture_output=True,
        text=True,
    )
    if torchvision_check.returncode != 0:
        print("Removing incompatible torchvision; PhoBERT training is text-only.")
        subprocess.run(
            [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision", "timm"],
            check=True,
        )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--upgrade",
        *HF_PKGS,
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable,
        "-c",
        "from transformers import AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, Trainer, TrainingArguments",
    ],
    check=True,
    env=SUBPROCESS_ENV,
)
import stf
MODEL_SOURCE = REPO_DIR / "src/stf/sentiment/model.py"
model_source = MODEL_SOURCE.read_text(encoding="utf-8")
if "save_only_model=True" not in model_source:
    raise RuntimeError(
        "Source branch develop chưa có bản sửa giảm dung lượng checkpoint. "
        "Hãy cập nhật branch rồi chạy lại notebook."
    )
print("Disk-safe CV source: enabled")

print("Repository ready:", REPO_DIR)
print("Package source:", stf.__file__)
print("Transformers runtime:", "transformers==5.15.1")

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU chưa được bật. Chọn T4 GPU rồi chạy lại cell này.")

In [ ]:
if Path("/kaggle").exists():
    DRIVE_ROOT = Path("/kaggle/working/stock-trend-experiments")
else:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/stock-trend-experiments")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print("Results will be saved under:", DRIVE_ROOT)

In [ ]:
KAGGLE_DATASET = "phuocthoai/stock-trend-forecasting"
ATTACHED_DIR = Path("/kaggle/input/stock-trend-forecasting")

if ATTACHED_DIR.exists():
    DATASET_DIR = ATTACHED_DIR
    print("Using attached Kaggle Dataset:", DATASET_DIR)
else:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True
    )
    import kagglehub

    DATASET_DIR = Path(kagglehub.dataset_download(KAGGLE_DATASET))
    print("Downloaded Kaggle Dataset to:", DATASET_DIR)

matches = sorted(DATASET_DIR.rglob("to_label_r1.csv"))
if not matches:
    raise FileNotFoundError("Kaggle Dataset không có to_label_r1.csv.")
DATA_PATH = matches[0]
print("Using label data:", DATA_PATH)

In [ ]:
import pandas as pd
from stf.sentiment.dataset import load_labeled

labeled = load_labeled(DATA_PATH)
print("Rows:", len(labeled))
print("Columns:", labeled.columns.tolist())
print("Label distribution:")
print(labeled["label"].value_counts().to_string())
assert {"title", "label"}.issubset(labeled.columns), (
    "Cần tệp có đủ title và label cuối đã được rà soát."
)
assert {"body", "body_preview"} & set(labeled.columns), (
    "Cần tệp có body hoặc body_preview cho các cấu hình context."
)
assert len(labeled) >= 301, "Cần ít nhất 301 bài đã được rà soát."

## Cấu hình chạy

Cell dưới đây tương đương với lệnh `sentiment-cv` trong README. Có thể thay `INPUT_VARIANT` và `TRUNCATION_STRATEGY` để chạy từng cấu hình. Giữ nguyên `SEED` giữa các cấu hình để so sánh công bằng.

In [ ]:
INPUT_VARIANT = "title_context"  # title | context | title_context
TRUNCATION_STRATEGY = "head_tail"  # head | tail | head_tail
FOLDS = 5
EPOCHS = 3
BATCH_SIZE = 16
SEED = 42
RUN_NAME = f"{INPUT_VARIANT}__{TRUNCATION_STRATEGY}__e{EPOCHS}"
OUTPUT_DIR = DRIVE_ROOT / RUN_NAME

command = [
    sys.executable,
    "-m",
    "stf.cli",
    "sentiment-cv",
    "--data",
    str(DATA_PATH),
    "--input-variant",
    INPUT_VARIANT,
    "--truncation-strategy",
    TRUNCATION_STRATEGY,
    "--folds",
    str(FOLDS),
    "--epochs",
    str(EPOCHS),
    "--batch-size",
    str(BATCH_SIZE),
    "--seed",
    str(SEED),
    "--output",
    str(OUTPUT_DIR),
]
print(" ".join(command))
subprocess.run(command, check=True, cwd=REPO_DIR, env=SUBPROCESS_ENV)

In [ ]:
import json

cv_json = OUTPUT_DIR / "cv_results.json"
cv_csv = OUTPUT_DIR / "cv_results.csv"
result = json.loads(cv_json.read_text(encoding="utf-8"))
folds = pd.read_csv(cv_csv)
print("Fold results:")
display(folds)
print("Aggregate:")
display(pd.DataFrame(result["aggregate"]).T)
print("Saved to:", OUTPUT_DIR)

## Chạy ma trận đầy đủ (tùy chọn)

Sau khi kiểm tra một cấu hình, có thể chạy 9 cấu hình (3 dạng đầu vào × 3 cách cắt). Mỗi cấu hình gồm 5 fold và 3 epoch nên cần nhiều thời gian; kết quả được lưu trực tiếp vào Drive.

In [ ]:
RUN_FULL_ABLATION = False
if RUN_FULL_ABLATION:
    command = [
        sys.executable,
        "-m",
        "stf.cli",
        "sentiment-ablation",
        "--data",
        str(DATA_PATH),
        "--folds",
        str(FOLDS),
        "--epochs",
        str(EPOCHS),
        "--batch-size",
        str(BATCH_SIZE),
        "--seed",
        str(SEED),
        "--output",
        str(DRIVE_ROOT / "ablation"),
    ]
    subprocess.run(command, check=True, cwd=REPO_DIR, env=SUBPROCESS_ENV)
else:
    print("Đang bỏ qua ma trận đầy đủ. Đổi RUN_FULL_ABLATION = True để chạy.")

## Diễn giải kết quả

- Chọn cấu hình theo `macro_f1` trung bình qua 5 fold; ghi cả độ lệch chuẩn.
- Kiểm tra F1 từng lớp và ma trận nhầm lẫn, không chỉ accuracy.
- Không đưa nhãn sơ bộ vào kết luận chính; chỉ dùng tệp đã có ít nhất 301 nhãn được người gán nhãn rà soát.
- Sau khi chọn mô hình cảm xúc, mới dùng checkpoint tốt nhất để suy luận toàn bộ tin Vietstock và xây dựng đặc trưng theo mã/ngày.